----
## <font color='CornflowerBlue'>Practical 5: Transformers and Protein Language Models</font> 
##### Alok Bharadwaj and Arjen Jakobi
---


> You shall know a word by the company it keeps.

**John Rupert Firth** a British linguist in *A Synopsis of Linguistic Theory* (1957)

The quote from John Firth above points to a unique aspect of human language. The meaning of a word can be inferred by the context in which it is commonly used. In any large corpus of text, words do not appear in any arbitrary pattern. They usually are found alongside other words. Being able to statistically model this context is a significant reason for the rapid development of Large Language Models in recent years. 

Proteins sequences, it turns out, also share broad similarities with language. Thus, many of the tools developed to model language have also been applied for protein sequences. In this week, we will look at language models as it applies to both text and proteins sequences. First, what is a protein sequence? 



### 5.1: Protein sequences

Proteins are long chains of amino acids. The order in which amino acids are arranged is called the protein's **primary structure**, and it encodes everything about the protein's three-dimensional shape and biological function.

Protein sequences are publicly accessible through the [UniProtKB](https://www.uniprot.org/) database.


#### Exercise 5.1.1: Protein sequences as language

Protein sequences are written using 20 standard single-letter amino acid codes (`A`, `C`, `D`, `E`, `F`, `G`, `H`, `I`, `K`, `L`, `M`, `N`, `P`, `Q`, `R`, `S`, `T`, `V`, `W`, `Y`). Because these codes are letters, some ordinary English words are also valid protein sub-sequences.

Use the code cell below to check which of the following words appear as exact sub-sequences in any database entry of the human proteome. 

| Word | Valid amino acid sequence? | Occurs in UniProt? |
|------|---------------------------|-------------------|
| `APPLE` | | |
| `PYTHON` | | |
| `DELFT` | | |
| `ANANAS` | | |
| `MACHINELEARNING` | | |
| `PIGTAIL` | | |
| `[YOUR IDEAS...]` | | |

In [ ]:
from src.utils import search_word_in_proteome
search_word_in_proteome("APPLE",context=5)

### 5.2 : Tokenisation:
A token is the basic unit of a language model. It is a numerical quantity that represents parts of a language. It could be a character, a word, parts of a word, or phrases. What makes a good token is that it captures rich contextual information but is also frequent enough to be useful to build a vocabulary. Let us explore some options to tokenise written texts. 

- **Character-level** — each character (including spaces) is its own token

  `[E][V][E][R][Y][·][C][H][A][R][A][C][T][E][R][·][I][S][·][A][·][T][O][K][E][N]`

- **Word-level** — each word is one token

  `[EVERY][·][WORD][·][IS][·][A][·][TOKEN]`

- **Sentence-level** — the whole sentence is one token

  `[EVERY SENTENCE IS A TOKEN]`

The balance between expressivity and representation forces us to choose between tokenisation schemes. Let us use the [tiny-shakespeare](https://huggingface.co/datasets/karpathy/tiny_shakespeare) dataset to make some observations. 

In [ ]:
with open("tinyshakespeare.txt", "r") as f:
    text = f.read()

We can first compute some statistics on our text corpus.

In [ ]:
# Character-level analysis
num_chars_total = len(text)
print(f"Total number of characters in the text: {num_chars_total}")

unique_chars = list(set(text))
num_unique_chars = len(unique_chars)
representation_ratio = num_chars_total / num_unique_chars

print(f"Number of unique characters in the text: {num_unique_chars}")
print(f"Representation ratio (total chars / unique chars): {representation_ratio:.2f}")
print(f"Unique characters in the text:")
print("-" * 40)
print(unique_chars)

To tokenise the text, we build two lookup dictionaries: one that maps each unique character to an integer index (**encode**), and one that maps each integer back to its character (**decode**). Together they define a reversible mapping between the raw text and its numerical representation — the vocabulary of this character-level tokeniser.

The encode step converts a string into a list of integers. The decode step inverts this, reconstructing the original string from a list of indices. Run the cell below to see this round-trip for the phrase `"to be, or"`.


In [ ]:
encode_tokens = {char: idx for idx, char in enumerate(unique_chars)}
decode_tokens = {idx: char for char, idx in encode_tokens.items()}

sample_text = "to be, or"
encoded_sample = [encode_tokens[char] for char in sample_text]
print(f"Encoded sample text: {encoded_sample}")
decoded_sample = "".join([decode_tokens[idx] for idx in encoded_sample])
print(f"Decoded sample text: {decoded_sample}")


Note that spaces, punctuation, and newline characters are all part of the character vocabulary. This is essential; without encoding whitespace, the model would lose track of word boundaries and sentence structure. The representation ratio of ~17,000 means each of the 65 unique characters appears on average 17,000 times in the corpus, giving the model ample training signal to learn each character's usage patterns.

Let us next look at some statistics at the level of words.

In [ ]:
# Word-level analysis
import random
words = text.split()
num_words_total = len(words)
print(f"Total number of words in the text: {num_words_total}")
unique_words = list(set(words))
num_unique_words = len(unique_words)
representation_ratio_words = num_words_total / num_unique_words

print(f"Number of unique words in the text: {num_unique_words}")
print(f"Representation ratio (total words / unique words): {representation_ratio_words:.2f}")
print(f"Some unique words in the text:")
print("-" * 40)
sample_words = random.sample(unique_words, 5)
print(sample_words)

We can do the same exercise as above and tokenise the text, but now we do the encoding at the level of words.

In [ ]:
encode_word_tokens = {word: idx for idx, word in enumerate(unique_words)}
decode_word_tokens = {idx: word for word, idx in encode_word_tokens.items()}

sample_word_text = "to be, or"
encoded_sample_words = [encode_word_tokens[word] for word in sample_word_text.split()]
print(f"Encoded sample word text: {encoded_sample_words}")
decoded_sample_words = " ".join([decode_word_tokens[idx] for idx in encoded_sample_words])
print(f"Decoded sample word text: {decoded_sample_words}")

And of course, we can also do it at the level of sentences.

In [ ]:
# Sentence-level analysis, assuming sentences are separated by newline.
sentences = text.split("\n")
num_sentences_total = len(sentences)
print(f"Total number of sentences in the text: {num_sentences_total}")
unique_sentences = list(set(sentences))
num_unique_sentences = len(unique_sentences)
representation_ratio_sentences = num_sentences_total / num_unique_sentences
print(f"Representation ratio (total sentences / unique sentences): {representation_ratio_sentences:.2f}")
print(f"Number of unique sentences in the text: {num_unique_sentences}")
print(f"Some unique sentences in the text:")
print("-" * 40)
sample_sentences = random.sample(unique_sentences, 3)
for sentence in sample_sentences:
    print(sentence.strip())
    print("-" * 40)


In [ ]:
encode_sentence_tokens = {sentence: idx for idx, sentence in enumerate(unique_sentences)}
decode_sentence_tokens = {idx: sentence for sentence, idx in encode_sentence_tokens.items()}

sample_sentence_text = "Why, Romeo, art thou mad?"
encoded_sample_sentence = encode_sentence_tokens[sample_sentence_text]
print(f"Encoded sample sentence text: {encoded_sample_sentence}")
decoded_sample_sentence = decode_sentence_tokens[encoded_sample_sentence]
print(f"Decoded sample sentence text: {decoded_sample_sentence.strip()}")


Comparing the three levels of tokenisation reveals a fundamental trade-off:

| Level | Vocabulary size | Representation ratio |
|-------|----------------|---------------------|
| Character | 65 | ~17,000 |
| Word | 25,670 | ~7.9 |
| Sentence | 25,722 | ~1.6 |

Characters are maximally compact: 65 symbols cover over a million characters, each appearing thousands of times. However, individual characters carry little meaning on their own; they must be grouped into words for semantic content to emerge.

Words are more expressive but inflate the vocabulary enormously and cannot handle unseen words (such as newly coined scientific terms or protein names). Sentences are the worst of both worlds: almost every sentence in the dataset is unique, so a sentence-level vocabulary barely generalises to new text.

A better strategy finds common **sub-word** units that sit between characters and words. Many words share the same root, so splitting them reduces vocabulary size while preserving expressivity. For example:

`["small", "smaller", "smallest", "big", "bigger", "biggest"]`

can be represented more compactly as:

`["small", "big", "##er", "##est"]`

The standard algorithm for learning these splits is **Byte-Pair Encoding (BPE)**, originally developed as a data-compression technique. BPE starts with individual characters and iteratively merges the most frequent adjacent pair of symbols until a target vocabulary size is reached. The result is a compact vocabulary of common sub-word units that generalises well to new text, including for example, scientific terminology it has never seen before.


In [ ]:
from src.lm_utils import get_bpe_encoder
enc = get_bpe_encoder("gpt2.tiktoken")
tokens_bpe = enc.encode(text)
unique_bpe_tokens = list(set(tokens_bpe))
num_bpe_tokens = len(tokens_bpe)
num_bpe_tokens_unique = len(unique_bpe_tokens)
representation_ratio_bpe = num_bpe_tokens / num_bpe_tokens_unique
print(f"Total number of BPE tokens in the text: {num_bpe_tokens}")
print(f"Number of unique BPE tokens in the text: {num_bpe_tokens_unique}")
print(f"Representation ratio (total BPE tokens / unique BPE tokens): {representation_ratio_bpe:.2f}")
print(f"Some unique BPE tokens in the text:")
print("-" * 40)
sample_bpe_tokens = random.sample(unique_bpe_tokens, 5)
decoded_bpe_tokens = [enc.decode([token]) for token in sample_bpe_tokens]
print(decoded_bpe_tokens)

In [ ]:
sample_text_bpe = "to be, or not to be"
encoded_sample_bpe = enc.encode(sample_text_bpe)
print(f"Encoded sample text (BPE tokens): {encoded_sample_bpe}")
decoded_sample_bpe = enc.decode(encoded_sample_bpe)
print(f"Decoded sample text from BPE tokens: {decoded_sample_bpe}")

#### From general language to sequence tokenisation
Tokenising protein sequences follows the same principles as language tokenisation. The standard approach is **residue-level tokenisation**: each amino acid is its own token. There are 20 standard amino acids plus a few additional special residues (selenocysteine, pyrrolysine), giving a vocabulary of around 22–30 tokens depending on the encoding scheme.

Residue-level tokenisation is particularly convenient from a modelling perspective. Unlike a single character in English, each amino acid already carries rich biochemical information such as defined size, charge, hydrophobicity, and reactivity. This makes it an expressive token on its own. Protein language models therefore work directly at the residue level without needing, or even avoiding, sub-word merging.

But what exactly do we mean by *features*? How can a model learn meaningful properties of amino acids purely from sequences? In the next section, we introduce **embeddings**. Embeddings are vectors that encode contextual meaning as numbers. This is how we can mathematically realise the quote by John Firth that opened this module: _a token shall be known by the company it keeps_.


### 5.3: Embeddings

An **embedding** is a vector of real numbers that represents a token in a way that is useful for a downstream task. The key distinction is between *meaningful* and *interpretable*: an embedding is meaningful if it helps the network make better predictions, but its individual dimensions need not correspond to any human-understandable concept.

In Week 3 you already encountered this idea. The node features used to describe atoms in a methanol molecule were a form of embedding: each dimension encoded some property (atom type, charge, etc.) that the graph network could exploit. Sequence embeddings work the same way, but the features are **learned from data** rather than hand-crafted.

A freshly initialised embedding is a random vector. After training on a large corpus, tokens that appear in similar contexts end up with similar embeddings — they are pushed closer together in vector space. This is the mathematical realisation of Firth's principle: *you shall know a word by the company it keeps*.

In the next module, we will see how the **attention mechanism**, the core of a transformer that underlies all modern large language models, dynamically adjusts these embeddings based on context. Interestingly, self-attention can be viewed as a special case of a Graph Neural Network where every token is connected to every other token, with connection strengths (attention weights) learned from the data.
